# Notebook 1: Training and Representation Metrics

Colab-ready workflow for cloning the GitHub repo, mounting Google Drive for checkpoints and artifacts, training or reusing a flow-circuits checkpoint, and inspecting P1/P2-facing representation metrics.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/jacobposchl/model-interpretability.git'
REPO_DIR = Path('/content/model_interpretability' if IN_COLAB else Path.cwd()).resolve()

if IN_COLAB:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)

    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/flow_circuits')
else:
    if not (REPO_DIR / 'flow_circuits').exists():
        raise RuntimeError('Run this notebook from the repository root or in Google Colab.')
    os.chdir(REPO_DIR)
    DRIVE_ROOT = REPO_DIR / 'artifacts' / 'local_notebook_runtime'

DATA_ROOT = DRIVE_ROOT / 'data'
EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments'
NOTEBOOK_ROOT = DRIVE_ROOT / 'notebook_runs'
for path in (DRIVE_ROOT, DATA_ROOT, EXPERIMENT_ROOT, NOTEBOOK_ROOT):
    path.mkdir(parents=True, exist_ok=True)

CONFIG_ROOT = REPO_DIR / 'configs' / 'flow'
print({'repo_dir': str(REPO_DIR), 'drive_root': str(DRIVE_ROOT), 'data_root': str(DATA_ROOT), 'experiment_root': str(EXPERIMENT_ROOT)})


In [ ]:
from flow_circuits.data import CIFAR10_STATS, build_cifar10_splits
from flow_circuits.evaluation import evaluate_representation_metrics
from flow_circuits.training import collect_model_outputs, load_components_from_checkpoint


In [ ]:
CONFIG_PATH = str(CONFIG_ROOT / 'resnet18_base.yaml')
CHECKPOINT_PATH = None
CIRCUITS_PATH = None
OUTPUT_DIR = str(NOTEBOOK_ROOT / 'nb01')
QUICK_MODE = True


In [ ]:
import copy
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLI_MODULES = {
    'flow-train': 'flow_circuits.cli.train',
    'flow-evaluate': 'flow_circuits.cli.evaluate',
}

def run_flow_cli(command: str, *args: str) -> None:
    try:
        subprocess.run([command, *args], check=True, cwd=REPO_DIR)
    except FileNotFoundError:
        subprocess.run([sys.executable, '-m', CLI_MODULES[command], *args], check=True, cwd=REPO_DIR)

with open(CONFIG_PATH, 'r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

config['data']['data_dir'] = str(DATA_ROOT / 'cifar10')
config['logging']['checkpoint_dir'] = str(EXPERIMENT_ROOT / config['experiment']['name'])

effective_config = copy.deepcopy(config)
if QUICK_MODE:
    effective_config['data']['batch_size'] = 32
    effective_config['data']['num_workers'] = 0
    effective_config['training']['phase_epochs'] = {'phase_a': 1, 'phase_b': 1, 'phase_c': 0}
    effective_config['training']['baseline_fit_images'] = 128
    effective_config['training']['baseline_eval_images'] = 128
    effective_config['training']['validation_images'] = 64
    effective_config['training']['alignment_max_pairs'] = 256
    effective_config['logging']['checkpoint_dir'] = str(OUTPUT_DIR / 'quick_checkpoints')

EFFECTIVE_CONFIG = OUTPUT_DIR / ('quick_config.yaml' if QUICK_MODE else 'effective_config.yaml')
EFFECTIVE_CONFIG.write_text(yaml.safe_dump(effective_config, sort_keys=False), encoding='utf-8')

EFFECTIVE_CHECKPOINT = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else Path(effective_config['logging']['checkpoint_dir']) / 'final.pt'
EVALUATION_PATH = OUTPUT_DIR / 'evaluation.json'
print({'config': str(EFFECTIVE_CONFIG), 'checkpoint': str(EFFECTIVE_CHECKPOINT), 'evaluation': str(EVALUATION_PATH)})


In [ ]:
if not EFFECTIVE_CHECKPOINT.exists():
    run_flow_cli('flow-train', '--config', str(EFFECTIVE_CONFIG))

run_flow_cli('flow-evaluate', '--checkpoint', str(EFFECTIVE_CHECKPOINT), '--output', str(EVALUATION_PATH))
evaluation_summary = json.loads(EVALUATION_PATH.read_text(encoding='utf-8'))
evaluation_summary


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
components = load_components_from_checkpoint(EFFECTIVE_CHECKPOINT, device=device)
loaders = build_cifar10_splits(
    data_dir=components.config['data']['data_dir'],
    batch_size=components.config['data']['batch_size'],
    num_workers=components.config['data'].get('num_workers', 0),
    seed=components.config['data'].get('seed', 0),
    augment_fit=components.config['data'].get('augment_fit', True),
    download=components.config['data'].get('download', True),
)
outputs = collect_model_outputs(
    components,
    loaders['test'],
    device=device,
    max_images=8 if QUICK_MODE else 32,
)
metrics = evaluate_representation_metrics(
    outputs['z'],
    outputs['flow_targets'],
    outputs['future_descriptors'],
    outputs['predicted_next'],
    outputs['reconstructed_current'],
    max_alignment_pairs=512 if QUICK_MODE else 2048,
)
{'representation_metrics': metrics.to_dict(), 'baseline_comparison': evaluation_summary['baseline_comparison']}


In [ ]:
mean = torch.tensor(CIFAR10_STATS['mean']).view(3, 1, 1)
std = torch.tensor(CIFAR10_STATS['std']).view(3, 1, 1)
sample_image = (outputs['images'][0].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
sample_label = int(outputs['labels'][0].item())
grid_size = components.tokenizer.grid_size
prediction_map = (outputs['predicted_next'][0, 0] * outputs['flow_targets'][0, 1]).sum(dim=-1).view(grid_size, grid_size).cpu().numpy()
reconstruction_map = (outputs['reconstructed_current'][0, 0] * outputs['flow_targets'][0, 0]).sum(dim=-1).view(grid_size, grid_size).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(sample_image)
axes[0].set_title(f'Sample image (label={sample_label})')
axes[0].axis('off')
pred_im = axes[1].imshow(prediction_map, cmap='viridis')
axes[1].set_title('Layer 1 next-step cosine')
recon_im = axes[2].imshow(reconstruction_map, cmap='magma')
axes[2].set_title('Layer 0 reconstruction cosine')
fig.colorbar(pred_im, ax=axes[1], fraction=0.046, pad=0.04)
fig.colorbar(recon_im, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()
